## Setup and config

In [1]:
!git clone https://github.com/huynoob1312/vietnamese_traffic_law_RAG.git
%cd vietnamese_traffic_law_RAG
!pip install -r requirements.txt

fatal: destination path 'vietnamese_traffic_law_RAG' already exists and is not an empty directory.
/content/vietnamese_traffic_law_RAG


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
!git pull

remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 16 (delta 12), reused 13 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (16/16), 1.81 KiB | 371.00 KiB/s, done.
From https://github.com/huynoob1312/vietnamese_traffic_law_RAG
   9257d63..834f937  main       -> origin/main
Updating 9257d63..834f937
error: Your local changes to the following files would be overwritten by merge:
	config.yaml
Please commit your changes or stash them before you merge.
Aborting


In [7]:
import os

# Đường dẫn thư mục gốc chứa Data trên Drive của bạn
DRIVE_DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/RAG_LEGAL"
# Đường dẫn repo code trên Colab
REPO_PATH = "/content/vietnamese_traffic_law_RAG"

if os.path.exists(DRIVE_DATA_PATH):
    print("Đã tìm thấy Google Drive. Bắt đầu chép dữ liệu...")

    # 1. Copy toàn bộ thư mục data (chứa các file luật)
    !cp -r "{DRIVE_DATA_PATH}/data" "{REPO_PATH}/"
    print("✅ Đã chép xong thư mục 'data/'")

    # 2. Copy TẤT CẢ các file benchmark (.jsonl) vào thư mục benchmark
    !cp "{DRIVE_DATA_PATH}/"*.jsonl "{REPO_PATH}/benchmark/" 2>/dev/null || true
    print("✅ Đã chép xong các file Benchmark (eval_dataset*.jsonl)")

else:
    print("❌ LỖI: Không tìm thấy đường dẫn trên Drive. Vui lòng kiểm tra lại DRIVE_DATA_PATH hoặc mount lại Drive!")


Đã tìm thấy Google Drive. Bắt đầu chép dữ liệu...
✅ Đã chép xong thư mục 'data/'
✅ Đã chép xong các file Benchmark (eval_dataset*.jsonl)


In [4]:
from google.colab import userdata
import os

os.environ['QDRANT_URL'] = userdata.get('QDRANT_URL')
os.environ['QDRANT_API_KEY'] = userdata.get('QDRANT_API_KEY')
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

print('Đã nạp thành công thông tin Qdrant Cloud!')

Đã nạp thành công thông tin Qdrant Cloud!


In [5]:
%%writefile config.yaml
qdrant:
  collection_name: "legal_docs"
  # collection_name: "legal_docs_bge"
  # collection_name: "legal_docs_sbert"
  # collection_name: "legal_docs_vietnamese_bi_encoder"

models:
  embedding: "intfloat/multilingual-e5-small"
  # embedding: "BAAI/bge-m3"
  # embedding: "keepitreal/vietnamese-sbert"
  # embedding: "bkai-foundation-models/vietnamese-bi-encoder"
  llm: "qwen2.5:3b"
chunking:
  thresh_dieu: 600
  thresh_khoan_default: 500
  min_chunk: 30
  min_chunk_diem: 5

retrieval:
  search_type: "hybrid" # "hybrid", "bm25", "vector"
  top_k: 10
  use_reranker: false
  use_multi_query: false
  reranker_model: "BAAI/bge-reranker-v2-m3"
  top_k_raw: 30

llm_params:
  provider: "gemini" # "gemini" or "ollama"
  model: "gemini-3.6-flash" # "gemini-3.6-flash" or "qwen2.5:3b"

benchmark:
  dataset_size: "full" # "sample" (200 câu) hoặc "full" (1500 câu)


Overwriting config.yaml


## Up to qdrant

In [6]:
# !python benchmark/ingest_data.py

## Benchmark Retrieval and embedding

### embedding: "intfloat/multilingual-e5-small"

In [8]:
!python benchmark/run_benchmark.py

modules.json: 100% 387/387 [00:00<00:00, 2.08MB/s]
README.md: 100% 498k/498k [00:00<00:00, 82.2MB/s]
sentence_bert_config.json: 100% 57.0/57.0 [00:00<00:00, 306kB/s]
config.json: 100% 655/655 [00:00<00:00, 3.79MB/s]

model.safetensors: downloading bytes:  38% 180M/471M [00:01<00:01, 220MB/s, 15.0MB/s  ]
model.safetensors: downloading bytes:  46% 215M/471M [00:01<00:01, 167MB/s, 19.2MB/s  ]
model.safetensors: downloading bytes:  67% 315M/471M [00:02<00:01, 144MB/s, 28.0MB/s  ]
model.safetensors: downloading bytes: 100% 315M/315M [00:02<00:00, 125MB/s, 27.8MB/s  ]
model.safetensors: reconstructing file: 100% 471M/471M [00:02<00:00, 187MB/s, 42.5MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 14365.54it/s]
tokenizer_config.json: 100% 443/443 [00:00<00:00, 1.85MB/s]

sentencepiece.bpe.model: downloading bytes:  35% 1.77M/5.07M [00:00<00:00, 4.11MB/s]
sentencepiece.bpe.model: downloading bytes: 100% 3.62M/3.62M [00:00<00:00, 8.11MB/s,  358kB/s  ]
sentencepiece.bpe.model: reconstructing 

In [9]:
%%writefile config.yaml
qdrant:
  collection_name: "legal_docs"
  # collection_name: "legal_docs_bge"
  # collection_name: "legal_docs_sbert"
  # collection_name: "legal_docs_vietnamese_bi_encoder"

models:
  embedding: "intfloat/multilingual-e5-small"
  # embedding: "BAAI/bge-m3"
  # embedding: "keepitreal/vietnamese-sbert"
  # embedding: "bkai-foundation-models/vietnamese-bi-encoder"
  llm: "qwen2.5:3b"
chunking:
  thresh_dieu: 600
  thresh_khoan_default: 500
  min_chunk: 30
  min_chunk_diem: 5

retrieval:
  search_type: "hybrid" # "hybrid", "bm25", "vector"
  top_k: 10
  use_reranker: false
  use_multi_query: false
  reranker_model: "BAAI/bge-reranker-v2-m3"
  top_k_raw: 30

llm_params:
  provider: "gemini" # "gemini" or "ollama"
  model: "gemini-3.6-flash" # "gemini-3.6-flash" or "qwen2.5:3b"

benchmark:
  dataset_size: "sample" # "sample" (200 câu) hoặc "full" (1500 câu)

Overwriting config.yaml


In [10]:
!python benchmark/run_benchmark.py

Loading weights: 100% 199/199 [00:00<00:00, 4656.26it/s]
BẮT ĐẦU CHẠY BENCHMARK TRÊN 197 CÂU HỎI...
Evaluating: 100% 197/197 [01:07<00:00,  2.93it/s]

KẾT QUẢ BENCHMARK (CẤU HÌNH HIỆN TẠI TỪ CONFIG.YAML)
R@10 (%): 83.25
R@100 (%): 83.25
MRR@10: 0.6097
MAP@10: 0.6097


### embedding: "BAAI/bge-m3"

In [13]:
%%writefile config.yaml
qdrant:
  # collection_name: "legal_docs"
  collection_name: "legal_docs_bge"
  # collection_name: "legal_docs_sbert"
  # collection_name: "legal_docs_vietnamese_bi_encoder"

models:
  # embedding: "intfloat/multilingual-e5-small"
  embedding: "BAAI/bge-m3"
  # embedding: "keepitreal/vietnamese-sbert"
  # embedding: "bkai-foundation-models/vietnamese-bi-encoder"
  llm: "qwen2.5:3b"
chunking:
  thresh_dieu: 600
  thresh_khoan_default: 500
  min_chunk: 30
  min_chunk_diem: 5

retrieval:
  search_type: "hybrid" # "hybrid", "bm25", "vector"
  top_k: 10
  use_reranker: false
  use_multi_query: false
  reranker_model: "BAAI/bge-reranker-v2-m3"
  top_k_raw: 30

llm_params:
  provider: "gemini" # "gemini" or "ollama"
  model: "gemini-3.6-flash" # "gemini-3.6-flash" or "qwen2.5:3b"

benchmark:
  dataset_size: "full" # "sample" (200 câu) hoặc "full" (1500 câu)


Overwriting config.yaml


In [14]:
!python benchmark/run_benchmark.py

modules.json: 100% 349/349 [00:00<00:00, 1.90MB/s]
config_sentence_transformers.json: 100% 123/123 [00:00<00:00, 696kB/s]
README.md: 100% 15.8k/15.8k [00:00<00:00, 33.2MB/s]
sentence_bert_config.json: 100% 54.0/54.0 [00:00<00:00, 364kB/s]
config.json: 100% 687/687 [00:00<00:00, 2.44MB/s]

pytorch_model.bin: downloading bytes:   6% 145M/2.27G [00:01<00:10, 211MB/s, 11.0MB/s  ]
pytorch_model.bin: downloading bytes:   8% 183M/2.27G [00:01<00:10, 202MB/s, 16.4MB/s  ]
pytorch_model.bin: downloading bytes:  67% 1.52G/2.27G [00:06<00:04, 185MB/s,  109MB/s  ]
pytorch_model.bin: downloading bytes:  67% 1.52G/2.27G [00:18<00:04, 185MB/s,  108MB/s  ]
pytorch_model.bin: downloading bytes: 100% 1.52G/1.52G [00:24<00:00, 62.5MB/s,  108MB/s  ]
pytorch_model.bin: reconstructing file: 100% 2.27G/2.27G [00:24<00:00, 93.3MB/s,  122MB/s  ]
Loading weights: 100% 391/391 [00:00<00:00, 12109.56it/s]

model.safetensors: downloading bytes:   0% 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 100% 444/444 [0